In [3]:
!pip install xgboost lightgbm catboost -q

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample = pd.read_csv('sample_submission.csv')

target = train.columns[-1]
id_col = sample.columns[0]

X = train.drop(columns=[target])
y = train[target]

if y.dtype == 'object':
    y = y.map({'Absence': 0, 'Presence': 1})

for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        test[col] = le.transform(test[col])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

test_xgb = np.zeros(len(test))
test_lgb = np.zeros(len(test))
test_cat = np.zeros(len(test))

for train_idx, valid_idx in skf.split(X, y):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    xgb = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )

    lgb = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        random_state=42
    )

    cat = CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        random_state=42
    )

    xgb.fit(X_train, y_train)
    lgb.fit(X_train, y_train)
    cat.fit(X_train, y_train)

    oof_xgb[valid_idx] = xgb.predict_proba(X_valid)[:,1]
    oof_lgb[valid_idx] = lgb.predict_proba(X_valid)[:,1]
    oof_cat[valid_idx] = cat.predict_proba(X_valid)[:,1]

    test_xgb += xgb.predict_proba(test)[:,1] / skf.n_splits
    test_lgb += lgb.predict_proba(test)[:,1] / skf.n_splits
    test_cat += cat.predict_proba(test)[:,1] / skf.n_splits

print("XGBoost LogLoss:", log_loss(y, oof_xgb))
print("LightGBM LogLoss:", log_loss(y, oof_lgb))
print("CatBoost LogLoss:", log_loss(y, oof_cat))

blend_oof = 0.4*oof_xgb + 0.3*oof_lgb + 0.3*oof_cat
blend_test = 0.4*test_xgb + 0.3*test_lgb + 0.3*test_cat

print("Blended LogLoss:", log_loss(y, blend_oof))

submission = pd.DataFrame({
    id_col: sample[id_col],
    sample.columns[1]: blend_test
})

submission.to_csv("submission.csv", index=False)
submission.head()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.4 MB/s eta 0:00:00
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.055980 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 677
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.056190 seconds.
You can set `force_row_wi

,id,Heart Disease
0,630000,0.924527
1,630001,0.007493
2,630002,0.983306
3,630003,0.005154
4,630004,0.224123
